# VoxIntel — 15: Semantic Error Risk Prediction

## Goal

This notebook upgrades VoxIntel from descriptive analysis to predictive analysis.

The central research question is:

> **Can taxonomy-based features predict downstream intent failure better than traditional ASR-style proxy metrics alone?**

By this stage, VoxIntel has already established that:

- baseline ASR harms downstream intent understanding severely
- fine-tuned ASR recovers a substantial portion of that loss
- some error categories are much more dangerous than others

This notebook tests whether those semantic categories carry predictive value beyond raw edit-volume style metrics.

---

## Important methodological distinction

The taxonomy features used in this notebook are **post-hoc**.
They are derived from alignment between:

- the ground-truth transcript
- and the ASR hypothesis

Therefore, this notebook evaluates the **explanatory and predictive value of semantic error categories under ground-truth-aware analysis**.

It does **not** claim that these exact taxonomy labels are available to a deployed system in real time.

So the contribution of Notebook 15 is:

> **post-hoc semantic error risk analysis**

not yet:

> deployable real-time failure prediction

A later stage can build a deployable risk model using only ASR-native signals such as:
- token confidence
- entropy
- N-best disagreement
- LM score
- acoustic uncertainty

That distinction is important for scientific honesty.

---

## Main experimental question

We compare three feature families:

### Model A — Proxy-only baseline
Uses coarse ASR-style features such as:
- total edit count
- substitution count
- insertion count
- deletion count
- lexical overlap

### Model B — Taxonomy-only model
Uses semantic features such as:
- primary error category
- number error flag
- proper-noun-like error flag
- entity-like error flag
- homophone-like error flag
- confidence

### Model C — Combined model
Uses both proxy and taxonomy features.

### Interaction ablation
We also separate:
- combined model without interactions
- combined model with interaction terms

so we can test whether cross-category interactions really add predictive value.

---

## Statistical additions in this final version

This notebook includes:
- explicit holdout evaluation
- 5-fold stratified cross-validation
- paired bootstrap confidence intervals for AUC differences
- calibration analysis
- random-feature sanity baseline
- heuristic SERS and learned SERS
- support-filtered intent risk ranking
- explicit limitations section


## Cell 0 — Make sure `src` is importable


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: c:\Users\ACER\OneDrive\Desktop\VoxIntel


## Cell 1 — Imports


In [2]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    brier_score_loss,
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.calibration import calibration_curve

sns.set_theme(style="whitegrid")
SEED = 42
rng = np.random.default_rng(SEED)

## Cell 2 — Load canonical post-consolidation datasets


In [3]:
REPORTS_DIR = PROJECT_ROOT / "reports"
paths = {
    "taxonomy": REPORTS_DIR / "error_taxonomy_dataset.csv",
    "impact": REPORTS_DIR / "semantic_impact_matrix.csv",
    "metric": REPORTS_DIR / "metric_comparison.csv",
}
for name, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

taxonomy_df = pd.read_csv(paths["taxonomy"])
semantic_impact_df = pd.read_csv(paths["impact"])
metric_comparison_df = pd.read_csv(paths["metric"])

print(taxonomy_df.shape)
display(metric_comparison_df)

(8688, 41)


,transcript_source,n_samples,accuracy,precision_macro,recall_macro,macro_f1,top3_accuracy,mean_confidence,median_confidence
0,ground_truth,8688,0.857735,0.696729,0.715841,0.703180,0.944406,0.922548,0.980201
1,baseline,8688,0.384899,0.404243,0.283904,0.308180,0.515654,0.734901,0.837562
2,finetuned,8688,0.734346,0.572839,0.570868,0.555436,0.845419,0.861017,0.968617


## Cell 3 — Create binary targets


In [4]:
taxonomy_df["baseline_intent_failed"] = (~taxonomy_df["baseline_correct"].astype(bool)).astype(int)
taxonomy_df["finetuned_intent_failed"] = (~taxonomy_df["finetuned_correct"].astype(bool)).astype(int)

display(taxonomy_df[[
    "intent", "scenario", "baseline_primary_error_type", "baseline_intent_failed",
    "finetuned_primary_error_type", "finetuned_intent_failed"
]].head())


,intent,scenario,baseline_primary_error_type,baseline_intent_failed,finetuned_primary_error_type,finetuned_intent_failed
0,qa_currency,qa,proper_noun_error,1,proper_noun_error,1
1,qa_currency,qa,proper_noun_error,1,proper_noun_error,1
2,email_query,email,proper_noun_error,1,no_error,0
3,email_query,email,proper_noun_error,1,no_error,0
4,email_query,email,proper_noun_error,1,no_error,0


## Cell 4 — Build the master modeling dataset

We include:
- proxy features
- taxonomy features
- interaction features
- confidence features

This creates one research table for all predictive experiments.


In [5]:
risk_df = taxonomy_df.copy()

risk_df["baseline_has_error"] = (risk_df["baseline_primary_error_type"] != "no_error").astype(int)
risk_df["finetuned_has_error"] = (risk_df["finetuned_primary_error_type"] != "no_error").astype(int)

risk_df["baseline_number_x_entity"] = (
    risk_df["baseline_contains_number_error"].astype(int) *
    risk_df["baseline_contains_entity_error"].astype(int)
)
risk_df["baseline_number_x_proper"] = (
    risk_df["baseline_contains_number_error"].astype(int) *
    risk_df["baseline_contains_proper_noun_error"].astype(int)
)
risk_df["finetuned_number_x_entity"] = (
    risk_df["finetuned_contains_number_error"].astype(int) *
    risk_df["finetuned_contains_entity_error"].astype(int)
)
risk_df["finetuned_number_x_proper"] = (
    risk_df["finetuned_contains_number_error"].astype(int) *
    risk_df["finetuned_contains_proper_noun_error"].astype(int)
)

for col in risk_df.columns:
    if col.startswith("baseline_contains_") or col.startswith("finetuned_contains_"):
        risk_df[col] = risk_df[col].astype(int)

for col in ["baseline_primary_error_type", "finetuned_primary_error_type", "intent", "scenario"]:
    risk_df[col] = risk_df[col].astype(str)

print(risk_df.shape)

(8688, 49)


## Cell 5 — Experimental protocol


In [6]:
TEST_SIZE = 0.2
N_SPLITS_CV = 5
BOOTSTRAP_SAMPLES = 2000
MIN_INTENT_SUPPORT = 30

protocol_df = pd.DataFrame([
    {"item": "holdout_split", "value": f"train_test_split(test_size={TEST_SIZE}, stratify=y, random_state={SEED})"},
    {"item": "cross_validation", "value": f"StratifiedKFold(n_splits={N_SPLITS_CV}, shuffle=True, random_state={SEED})"},
    {"item": "bootstrap", "value": f"paired bootstrap CI with n_boot={BOOTSTRAP_SAMPLES}"},
    {"item": "intent_support_filter", "value": MIN_INTENT_SUPPORT},
])

display(protocol_df)


,item,value
0,holdout_split,"train_test_split(test_size=0.2, stratify=y, ra..."
1,cross_validation,"StratifiedKFold(n_splits=5, shuffle=True, rand..."
2,bootstrap,paired bootstrap CI with n_boot=2000
3,intent_support_filter,30


## Cell 6 — Limitation on independent evaluation

This notebook trains and evaluates the risk models using splits derived from the current canonical dataset.
That is acceptable for **exploratory research modeling**, but it is not the same as a fully locked external evaluation.

So the correct interpretation is:

- **now**: exploratory / research-phase predictive analysis on the current canonical pool
- **later**: final locked evaluation on an official unseen test set

Also, if speaker IDs are not available here, we cannot perform speaker-grouped cross-validation. That should be stated explicitly in the final paper/report.


## Cell 7 — Define feature groups


In [7]:
proxy_features_baseline = [
    "baseline_n_total_edits",
    "baseline_n_substitutions",
    "baseline_n_insertions",
    "baseline_n_deletions",
    "baseline_lexical_overlap",
]
proxy_features_finetuned = [
    "finetuned_n_total_edits",
    "finetuned_n_substitutions",
    "finetuned_n_insertions",
    "finetuned_n_deletions",
    "finetuned_lexical_overlap",
]

taxonomy_features_baseline = [
    "baseline_primary_error_type",
    "baseline_contains_number_error",
    "baseline_contains_proper_noun_error",
    "baseline_contains_entity_error",
    "baseline_contains_homophone_like_error",
    "baseline_confidence",
]
taxonomy_features_finetuned = [
    "finetuned_primary_error_type",
    "finetuned_contains_number_error",
    "finetuned_contains_proper_noun_error",
    "finetuned_contains_entity_error",
    "finetuned_contains_homophone_like_error",
    "finetuned_confidence",
]

interaction_features_baseline = [
    "baseline_number_x_entity",
    "baseline_number_x_proper",
]
interaction_features_finetuned = [
    "finetuned_number_x_entity",
    "finetuned_number_x_proper",
]

combined_no_interaction_baseline = proxy_features_baseline + taxonomy_features_baseline
combined_no_interaction_finetuned = proxy_features_finetuned + taxonomy_features_finetuned
combined_full_baseline = combined_no_interaction_baseline + interaction_features_baseline
combined_full_finetuned = combined_no_interaction_finetuned + interaction_features_finetuned


## Cell 8 — Modeling helpers


In [8]:
def split_feature_types(df, features):
    numeric_features = [c for c in features if pd.api.types.is_numeric_dtype(df[c])]
    categorical_features = [c for c in features if c not in numeric_features]
    return numeric_features, categorical_features


def build_preprocessor(df, features):
    numeric_features, categorical_features = split_feature_types(df, features)
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    return ColumnTransformer([
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ])


def safe_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "brier": brier_score_loss(y_true, y_prob),
    }


def build_pipeline(df, features, estimator):
    return Pipeline([
        ("preprocess", build_preprocessor(df, features)),
        ("model", estimator),
    ])


def run_holdout_experiment(df, feature_cols, target_col, model_name, estimator):
    X = df[feature_cols].copy()
    y = df[target_col].astype(int).copy()
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
    )
    model = build_pipeline(X_train, feature_cols, estimator)
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    result = {
        "model_name": model_name,
        "target": target_col,
        "n_train": int(len(X_train)),
        "n_test": int(len(X_test)),
        **safe_metrics(y_test, y_prob),
    }
    return model, result, X_test, y_test, y_prob


def run_cv_auc(df, feature_cols, target_col, estimator):
    X = df[feature_cols].copy()
    y = df[target_col].astype(int).copy()
    cv = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=SEED)
    aucs = []
    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        model = build_pipeline(X_train, feature_cols, estimator)
        model.fit(X_train, y_train)
        y_prob = model.predict_proba(X_test)[:, 1]
        aucs.append(roc_auc_score(y_test, y_prob))
    return float(np.mean(aucs)), float(np.std(aucs))


def bootstrap_auc_difference(y_true, prob_a, prob_b, n_boot=BOOTSTRAP_SAMPLES, seed=SEED):
    rng_local = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    prob_a = np.asarray(prob_a)
    prob_b = np.asarray(prob_b)
    diffs = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng_local.integers(0, n, n)
        yb = y_true[idx]
        if len(np.unique(yb)) < 2:
            continue
        diffs.append(roc_auc_score(yb, prob_b[idx]) - roc_auc_score(yb, prob_a[idx]))
    diffs = np.array(diffs)
    return {
        "mean_auc_difference": float(np.mean(diffs)),
        "ci_low": float(np.quantile(diffs, 0.025)),
        "ci_high": float(np.quantile(diffs, 0.975)),
        "fraction_positive": float((diffs > 0).mean()),
        "n_boot_used": int(len(diffs)),
    }


## Cell 9 — Random-feature sanity baseline


In [9]:
risk_df["random_noise_1"] = rng.normal(size=len(risk_df))
risk_df["random_noise_2"] = rng.normal(size=len(risk_df))
risk_df["random_noise_3"] = rng.normal(size=len(risk_df))
random_feature_set = ["random_noise_1", "random_noise_2", "random_noise_3"]
random_baseline_model, random_baseline_result, _, _, _ = run_holdout_experiment(
    risk_df, random_feature_set, "baseline_intent_failed", "logreg_random_baseline", LogisticRegression(max_iter=2000, random_state=SEED)
)
pd.DataFrame([random_baseline_result])


,model_name,target,n_train,n_test,roc_auc,pr_auc,f1,precision,recall,brier
0,logreg_random_baseline,baseline_intent_failed,6950,1738,0.49219,0.603185,0.761667,0.615075,1.0,0.237159


## Cell 10 — Holdout experiments: proxy-only, taxonomy-only, combined-no-interaction, combined-full


In [10]:
def run_family(df, features, target, family_label):
    out = []
    models = [
        (f"logreg_{family_label}", LogisticRegression(max_iter=2000, random_state=SEED)),
        (f"rf_{family_label}", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)),
    ]
    store = {}
    for model_name, est in models:
        model, result, X_test, y_test, y_prob = run_holdout_experiment(df, features, target, model_name, est)
        out.append(result)
        store[model_name] = (model, X_test, y_test, y_prob)
    return out, store

results = [random_baseline_result]
stores = {}

for target, proxy_feats, tax_feats, comb_no_int, comb_full in [
    ("baseline_intent_failed", proxy_features_baseline, taxonomy_features_baseline, combined_no_interaction_baseline, combined_full_baseline),
    ("finetuned_intent_failed", proxy_features_finetuned, taxonomy_features_finetuned, combined_no_interaction_finetuned, combined_full_finetuned),
]:
    r1, s1 = run_family(risk_df, proxy_feats, target, "proxy_only")
    r2, s2 = run_family(risk_df, tax_feats, target, "taxonomy_only")
    r3, s3 = run_family(risk_df, comb_no_int, target, "combined_no_interaction")
    r4, s4 = run_family(risk_df, comb_full, target, "combined_full")
    results.extend(r1 + r2 + r3 + r4)
    stores.update(s1); stores.update(s2); stores.update(s3); stores.update(s4)

model_comparison_df = pd.DataFrame(results).sort_values(["target", "roc_auc", "pr_auc"], ascending=[True, False, False]).reset_index(drop=True)
display(model_comparison_df)


,model_name,target,n_train,n_test,roc_auc,pr_auc,f1,precision,recall,brier
0,logreg_combined_full,baseline_intent_failed,6950,1738,0.840839,0.880966,0.824666,0.811594,0.838167,0.154544
1,logreg_combined_no_interaction,baseline_intent_failed,6950,1738,0.840827,0.880949,0.824666,0.811594,0.838167,0.154550
2,rf_combined_full,baseline_intent_failed,6950,1738,0.840751,0.873598,0.823422,0.805730,0.841908,0.159107
3,rf_combined_no_interaction,baseline_intent_failed,6950,1738,0.840619,0.874126,0.820183,0.804680,0.836296,0.159394
4,logreg_taxonomy_only,baseline_intent_failed,6950,1738,0.805753,0.848710,0.808472,0.780000,0.839102,0.175475
5,logreg_proxy_only,baseline_intent_failed,6950,1738,0.795079,0.846445,0.796595,0.764402,0.831618,0.179508
6,rf_proxy_only,baseline_intent_failed,6950,1738,0.784072,0.834469,0.778499,0.771350,0.785781,0.183649
7,rf_taxonomy_only,baseline_intent_failed,6950,1738,0.755600,0.802597,0.754159,0.745205,0.763330,0.225815
8,logreg_random_baseline,baseline_intent_failed,6950,1738,0.492190,0.603185,0.761667,0.615075,1.000000,0.237159
9,rf_combined_no_interaction,finetuned_intent_failed,6950,1738,0.915214,0.816639,0.747253,0.758929,0.735931,0.098706


## Cell 11 — Cross-validation AUC estimates


In [11]:
cv_rows = []
cv_experiments = [
    ("baseline_intent_failed", "proxy_logreg", proxy_features_baseline),
    ("baseline_intent_failed", "taxonomy_logreg", taxonomy_features_baseline),
    ("baseline_intent_failed", "combined_no_interaction_logreg", combined_no_interaction_baseline),
    ("baseline_intent_failed", "combined_full_logreg", combined_full_baseline),
    ("finetuned_intent_failed", "proxy_logreg", proxy_features_finetuned),
    ("finetuned_intent_failed", "taxonomy_logreg", taxonomy_features_finetuned),
    ("finetuned_intent_failed", "combined_no_interaction_logreg", combined_no_interaction_finetuned),
    ("finetuned_intent_failed", "combined_full_logreg", combined_full_finetuned),
]
for target, label, features in cv_experiments:
    mean_auc, std_auc = run_cv_auc(risk_df, features, target, LogisticRegression(max_iter=2000, random_state=SEED))
    cv_rows.append({"target": target, "model_name": label, "cv_roc_auc_mean": mean_auc, "cv_roc_auc_std": std_auc})
cv_summary_df = pd.DataFrame(cv_rows)
display(cv_summary_df)


,target,model_name,cv_roc_auc_mean,cv_roc_auc_std
0,baseline_intent_failed,proxy_logreg,0.797987,0.008535
1,baseline_intent_failed,taxonomy_logreg,0.799624,0.012029
2,baseline_intent_failed,combined_no_interaction_logreg,0.841842,0.009605
3,baseline_intent_failed,combined_full_logreg,0.841840,0.009619
4,finetuned_intent_failed,proxy_logreg,0.699187,0.010874
5,finetuned_intent_failed,taxonomy_logreg,0.817495,0.010369
6,finetuned_intent_failed,combined_no_interaction_logreg,0.833779,0.009071
7,finetuned_intent_failed,combined_full_logreg,0.833704,0.009120


## Cell 12 — Paired bootstrap confidence intervals for AUC differences


In [12]:
# Compare proxy-only vs combined-full logistic regression on each target
bootstrap_rows = []
for target in ["baseline_intent_failed", "finetuned_intent_failed"]:
    if target == "baseline_intent_failed":
        proxy_key = "logreg_proxy_only"
        full_key = "logreg_combined_full"
    else:
        proxy_key = "logreg_proxy_only"
        full_key = "logreg_combined_full"

# Retrieve stored predictions by target via matching rows instead of direct dict collision-safe naming.
# Re-run the necessary models deterministically on the same split for clean paired comparison.

def paired_probs(df, feature_a, feature_b, target_col):
    X_a = df[feature_a].copy()
    X_b = df[feature_b].copy()
    y = df[target_col].astype(int).copy()
    Xa_train, Xa_test, y_train, y_test = train_test_split(X_a, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
    Xb_train, Xb_test, _, _ = train_test_split(X_b, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
    model_a = build_pipeline(Xa_train, feature_a, LogisticRegression(max_iter=2000, random_state=SEED))
    model_b = build_pipeline(Xb_train, feature_b, LogisticRegression(max_iter=2000, random_state=SEED))
    model_a.fit(Xa_train, y_train)
    model_b.fit(Xb_train, y_train)
    pa = model_a.predict_proba(Xa_test)[:, 1]
    pb = model_b.predict_proba(Xb_test)[:, 1]
    return y_test.to_numpy(), pa, pb

for target_col, proxy_feats, full_feats in [
    ("baseline_intent_failed", proxy_features_baseline, combined_full_baseline),
    ("finetuned_intent_failed", proxy_features_finetuned, combined_full_finetuned),
]:
    y_test, pa, pb = paired_probs(risk_df, proxy_feats, full_feats, target_col)
    row = bootstrap_auc_difference(y_test, pa, pb)
    row["target"] = target_col
    row["comparison"] = "combined_full_vs_proxy_logreg"
    bootstrap_rows.append(row)

bootstrap_significance_df = pd.DataFrame(bootstrap_rows)
display(bootstrap_significance_df)


,mean_auc_difference,ci_low,ci_high,fraction_positive,n_boot_used,target,comparison
0,0.045872,0.032796,0.059688,1.0,2000,baseline_intent_failed,combined_full_vs_proxy_logreg
1,0.134036,0.109003,0.156336,1.0,2000,finetuned_intent_failed,combined_full_vs_proxy_logreg


## Cell 13 — Calibration analysis


In [13]:
def calibration_table(y_true, y_prob, n_bins=10):
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy="quantile")
    return pd.DataFrame({"predicted": prob_pred, "observed": prob_true})

# Reuse paired comparison probabilities from previous cell for clean calibration comparison
baseline_y, baseline_proxy_prob, baseline_comb_prob = paired_probs(risk_df, proxy_features_baseline, combined_full_baseline, "baseline_intent_failed")
finetuned_y, finetuned_proxy_prob, finetuned_comb_prob = paired_probs(risk_df, proxy_features_finetuned, combined_full_finetuned, "finetuned_intent_failed")

calibration_baseline_proxy_df = calibration_table(baseline_y, baseline_proxy_prob)
calibration_baseline_combined_df = calibration_table(baseline_y, baseline_comb_prob)
calibration_finetuned_proxy_df = calibration_table(finetuned_y, finetuned_proxy_prob)
calibration_finetuned_combined_df = calibration_table(finetuned_y, finetuned_comb_prob)

display(calibration_baseline_proxy_df)
display(calibration_baseline_combined_df)


,predicted,observed
0,0.116614,0.250000
1,0.314105,0.284264
2,0.440582,0.358824
3,0.529849,0.546053
4,0.629573,0.614943
5,0.715255,0.690722
6,0.774982,0.748387
7,0.821549,0.848837
8,0.873462,0.937198
9,0.883713,0.907801


,predicted,observed
0,0.129286,0.132184
1,0.216577,0.229885
2,0.341450,0.327586
3,0.480474,0.462428
4,0.614619,0.683908
5,0.728141,0.764368
6,0.804082,0.809249
7,0.875566,0.879310
8,0.929487,0.931034
9,0.967306,0.931034


## Cell 14 — Feature importance: logistic regression


In [14]:
def extract_logreg_importance_from_features(df, features, target_col):
    X = df[features].copy()
    y = df[target_col].astype(int).copy()
    model = build_pipeline(X, features, LogisticRegression(max_iter=2000, random_state=SEED))
    model.fit(X, y)
    preprocessor = model.named_steps["preprocess"]
    clf = model.named_steps["model"]
    feature_names = preprocessor.get_feature_names_out()
    coefs = clf.coef_[0]
    return pd.DataFrame({"feature": feature_names, "coefficient": coefs, "abs_coefficient": np.abs(coefs)}).sort_values("abs_coefficient", ascending=False).reset_index(drop=True)

logreg_feature_importance_baseline_df = extract_logreg_importance_from_features(risk_df, combined_full_baseline, "baseline_intent_failed")
logreg_feature_importance_finetuned_df = extract_logreg_importance_from_features(risk_df, combined_full_finetuned, "finetuned_intent_failed")

display(logreg_feature_importance_baseline_df.head(20))
display(logreg_feature_importance_finetuned_df.head(20))


,feature,coefficient,abs_coefficient
0,cat__baseline_primary_error_type_entity_error,-1.530535,1.530535
1,cat__baseline_primary_error_type_no_error,1.234317,1.234317
2,num__baseline_lexical_overlap,-1.107169,1.107169
3,num__baseline_confidence,-0.938840,0.938840
4,cat__baseline_primary_error_type_deletion,0.342038,0.342038
5,cat__baseline_primary_error_type_proper_noun_e...,0.230521,0.230521
6,num__baseline_contains_entity_error,0.110064,0.110064
7,num__baseline_contains_proper_noun_error,-0.095918,0.095918
8,num__baseline_contains_homophone_like_error,-0.084541,0.084541
9,num__baseline_n_insertions,-0.034244,0.034244


,feature,coefficient,abs_coefficient
0,num__finetuned_confidence,-1.124961,1.124961
1,cat__finetuned_primary_error_type_entity_error,-1.093820,1.093820
2,cat__finetuned_primary_error_type_no_error,0.908003,0.908003
3,num__finetuned_lexical_overlap,-0.782260,0.782260
4,cat__finetuned_primary_error_type_proper_noun_...,-0.506837,0.506837
5,num__finetuned_contains_entity_error,0.225720,0.225720
6,num__finetuned_contains_homophone_like_error,-0.094220,0.094220
7,cat__finetuned_primary_error_type_deletion,0.061896,0.061896
8,num__finetuned_contains_proper_noun_error,-0.055988,0.055988
9,num__finetuned_n_deletions,-0.055025,0.055025


## Cell 15 — Feature importance: random forest


In [15]:
def extract_rf_importance_from_features(df, features, target_col):
    X = df[features].copy()
    y = df[target_col].astype(int).copy()
    model = build_pipeline(X, features, RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1))
    model.fit(X, y)
    preprocessor = model.named_steps["preprocess"]
    rf = model.named_steps["model"]
    feature_names = preprocessor.get_feature_names_out()
    return pd.DataFrame({"feature": feature_names, "importance": rf.feature_importances_}).sort_values("importance", ascending=False).reset_index(drop=True)

rf_feature_importance_baseline_df = extract_rf_importance_from_features(risk_df, combined_full_baseline, "baseline_intent_failed")
rf_feature_importance_finetuned_df = extract_rf_importance_from_features(risk_df, combined_full_finetuned, "finetuned_intent_failed")

display(rf_feature_importance_baseline_df.head(20))
display(rf_feature_importance_finetuned_df.head(20))


,feature,importance
0,num__baseline_confidence,0.578357
1,num__baseline_lexical_overlap,0.253618
2,num__baseline_n_substitutions,0.056971
3,num__baseline_n_total_edits,0.048737
4,num__baseline_contains_homophone_like_error,0.016527
5,num__baseline_n_deletions,0.010469
6,num__baseline_contains_entity_error,0.010349
7,num__baseline_n_insertions,0.007477
8,num__baseline_contains_proper_noun_error,0.006309
9,cat__baseline_primary_error_type_proper_noun_e...,0.003291


,feature,importance
0,num__finetuned_confidence,0.740471
1,num__finetuned_lexical_overlap,0.162059
2,num__finetuned_n_substitutions,0.031201
3,num__finetuned_n_total_edits,0.028089
4,num__finetuned_contains_homophone_like_error,0.014365
5,num__finetuned_n_deletions,0.005375
6,num__finetuned_contains_entity_error,0.004851
7,num__finetuned_n_insertions,0.004800
8,cat__finetuned_primary_error_type_proper_noun_...,0.002156
9,num__finetuned_contains_proper_noun_error,0.002085


## Cell 16 — Heuristic SERS and out-of-fold learned SERS


In [16]:
risk_df["SERS_H_baseline"] = (
    0.35 * risk_df["baseline_contains_number_error"].astype(float) +
    0.30 * risk_df["baseline_contains_proper_noun_error"].astype(float) +
    0.20 * risk_df["baseline_contains_entity_error"].astype(float) +
    0.10 * (risk_df["baseline_n_deletions"] > 0).astype(float) +
    0.05 * (1.0 - risk_df["baseline_lexical_overlap"].fillna(1.0).astype(float))
)
risk_df["SERS_H_finetuned"] = (
    0.35 * risk_df["finetuned_contains_number_error"].astype(float) +
    0.30 * risk_df["finetuned_contains_proper_noun_error"].astype(float) +
    0.20 * risk_df["finetuned_contains_entity_error"].astype(float) +
    0.10 * (risk_df["finetuned_n_deletions"] > 0).astype(float) +
    0.05 * (1.0 - risk_df["finetuned_lexical_overlap"].fillna(1.0).astype(float))
)

# Learned SERS via out-of-fold probabilities (OOF)
cv = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=SEED)

baseline_oof_model = build_pipeline(risk_df, combined_full_baseline, LogisticRegression(max_iter=2000, random_state=SEED))
risk_df["SERS_L_baseline"] = cross_val_predict(
    baseline_oof_model,
    risk_df[combined_full_baseline],
    risk_df["baseline_intent_failed"],
    cv=cv,
    method="predict_proba",
    n_jobs=None,
)[:, 1]

finetuned_oof_model = build_pipeline(risk_df, combined_full_finetuned, LogisticRegression(max_iter=2000, random_state=SEED))
risk_df["SERS_L_finetuned"] = cross_val_predict(
    finetuned_oof_model,
    risk_df[combined_full_finetuned],
    risk_df["finetuned_intent_failed"],
    cv=cv,
    method="predict_proba",
    n_jobs=None,
)[:, 1]

display(risk_df[["SERS_H_baseline", "SERS_L_baseline", "SERS_H_finetuned", "SERS_L_finetuned"]].describe())


,SERS_H_baseline,SERS_L_baseline,SERS_H_finetuned,SERS_L_finetuned
count,8688.000000,8688.000000,8688.000000,8688.000000
mean,0.520392,0.614926,0.417929,0.265626
std,0.165638,0.283986,0.230626,0.253427
min,0.000000,0.015570,0.000000,0.010735
25%,0.525000,0.361056,0.507923,0.103181
50%,0.537500,0.677748,0.519444,0.142997
75%,0.550000,0.873835,0.528571,0.327806
max,0.996875,0.989603,0.982353,0.993885


## Cell 17 — Validate SERS bins


In [17]:
def score_bin_analysis(scores, failures, system, score_name):
    tmp = pd.DataFrame({"score": scores, "failed": failures})
    tmp["bin"] = pd.qcut(tmp["score"], q=3, labels=["low", "medium", "high"], duplicates="drop")
    out = tmp.groupby("bin").agg(samples=("score", "size"), failure_rate=("failed", "mean"), mean_score=("score", "mean")).reset_index()
    out["system"] = system
    out["score_name"] = score_name
    return out

semantic_error_risk_score_bins_df = pd.concat([
    score_bin_analysis(risk_df["SERS_H_baseline"], risk_df["baseline_intent_failed"], "baseline", "SERS_H"),
    score_bin_analysis(risk_df["SERS_L_baseline"], risk_df["baseline_intent_failed"], "baseline", "SERS_L_OOF"),
    score_bin_analysis(risk_df["SERS_H_finetuned"], risk_df["finetuned_intent_failed"], "finetuned", "SERS_H"),
    score_bin_analysis(risk_df["SERS_L_finetuned"], risk_df["finetuned_intent_failed"], "finetuned", "SERS_L_OOF"),
], ignore_index=True)

display(semantic_error_risk_score_bins_df)


C:\Users\ACER\AppData\Local\Temp\ipykernel_12164\1956591693.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = tmp.groupby("bin").agg(samples=("score", "size"), failure_rate=("failed", "mean"), mean_score=("score", "mean")).reset_index()
C:\Users\ACER\AppData\Local\Temp\ipykernel_12164\1956591693.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = tmp.groupby("bin").agg(samples=("score", "size"), failure_rate=("failed", "mean"), mean_score=("score", "mean")).reset_index()
C:\Users\ACER\AppData\Local\Temp\ipykernel_12164\1956591693.py:4: FutureWarning: The default of observed=False is deprecated a

,bin,samples,failure_rate,mean_score,system,score_name
0,low,2902,0.334252,0.406424,baseline,SERS_H
1,medium,2990,0.680268,0.537400,baseline,SERS_H
2,high,2796,0.836910,0.620492,baseline,SERS_H
3,low,2896,0.254144,0.262390,baseline,SERS_L_OOF
4,medium,2896,0.684047,0.667679,baseline,SERS_L_OOF
5,high,2896,0.907113,0.914708,baseline,SERS_L_OOF
6,low,2910,0.145017,0.165877,finetuned,SERS_H
7,medium,3158,0.223876,0.519664,finetuned,SERS_H
8,high,2620,0.450000,0.575257,finetuned,SERS_H
9,low,2896,0.072859,0.086311,finetuned,SERS_L_OOF


## Cell 18 — Intent-level risk rankings with support filter


In [18]:
intent_risk_baseline_df = risk_df.groupby("intent").agg(
    n_samples=("intent", "size"),
    avg_risk=("SERS_L_baseline", "mean"),
    failure_rate=("baseline_intent_failed", "mean"),
).reset_index().sort_values(["avg_risk", "n_samples"], ascending=[False, False])

intent_risk_finetuned_df = risk_df.groupby("intent").agg(
    n_samples=("intent", "size"),
    avg_risk=("SERS_L_finetuned", "mean"),
    failure_rate=("finetuned_intent_failed", "mean"),
).reset_index().sort_values(["avg_risk", "n_samples"], ascending=[False, False])

intent_risk_baseline_supported_df = intent_risk_baseline_df[intent_risk_baseline_df["n_samples"] >= MIN_INTENT_SUPPORT].copy()
intent_risk_finetuned_supported_df = intent_risk_finetuned_df[intent_risk_finetuned_df["n_samples"] >= MIN_INTENT_SUPPORT].copy()

display(intent_risk_baseline_supported_df.head(20))
display(intent_risk_finetuned_supported_df.head(20))


,intent,n_samples,avg_risk,failure_rate
5,audio_volume_up,101,0.737304,0.663366
28,iot_hue_lightdim,79,0.736455,0.759494
1,alarm_remove,87,0.734362,0.862069
4,audio_volume_mute,89,0.714754,0.741573
25,iot_cleaning,80,0.704242,0.737500
15,email_query,282,0.702180,0.851064
29,iot_hue_lightoff,55,0.701616,0.763636
63,takeaway_order,77,0.699489,0.636364
66,transport_taxi,97,0.696207,0.721649
3,audio_volume_down,95,0.695410,0.642105


,intent,n_samples,avg_risk,failure_rate
29,iot_hue_lightoff,55,0.623877,0.872727
68,transport_traffic,87,0.519470,0.977011
53,qa_maths,54,0.480413,0.518519
28,iot_hue_lightdim,79,0.477134,0.265823
26,iot_coffee,68,0.451729,0.367647
21,general_quirky,438,0.391443,0.488584
61,social_post,185,0.386071,0.367568
25,iot_cleaning,80,0.375935,0.437500
11,cooking_recipe,183,0.349813,0.393443
62,social_query,83,0.345915,0.698795


## Cell 19 — Highest-confidence wrong predictions


In [19]:
high_confidence_wrong_df = taxonomy_df[
    (~taxonomy_df["baseline_correct"]) & (taxonomy_df["baseline_confidence"] >= taxonomy_df["baseline_confidence"].quantile(0.9))
].copy()

high_confidence_wrong_df = high_confidence_wrong_df[[
    "intent", "scenario", "ground_truth", "baseline_transcript", "finetuned_transcript",
    "baseline_primary_error_type", "baseline_confidence", "baseline_alignment_summary",
    "baseline_impact", "baseline_correct", "finetuned_correct",
]].sort_values("baseline_confidence", ascending=False)

display(high_confidence_wrong_df.head(30))


,intent,scenario,ground_truth,baseline_transcript,finetuned_transcript,baseline_primary_error_type,baseline_confidence,baseline_alignment_summary,baseline_impact,baseline_correct,finetuned_correct
8279,calendar_remove,calendar,please delete all reminder of sunday,PRESENLLY ALL REMIND HER OF SUNDAY,PLEASE DELETE AL REMINDER OF SUNDAY,proper_noun_error,0.996560,replace:please delete->presenlly | replace:rem...,high_impact,False,True
2253,qa_factoid,qa,the following question asks you to analyze tea...,THE FORTER QUESTION ASKED YOU TO ANALYZE TEACH...,THE FOLOWING QUESTION ASK YOU TO ANALYZE TOU T...,proper_noun_error,0.996464,replace:following->forter | replace:asks->aske...,high_impact,False,False
8086,alarm_query,alarm,how many alarms do i have set for morning hour...,ARE YOU ARMSA HAVE SAID FOR MORNING HOUR BETWE...,HOW MANY ALARMS DO I HAVE SET FOR MORNING HOUR...,number_error,0.996348,replace:how many alarms do i->are you armsa | ...,high_impact,False,True
7085,calendar_remove,calendar,delete the next event on the calendar,TO LEAD THE NEXT EVENT ON THE CALENDAR,DELETE THE NEXT EVENT ON THE CALENDAR,proper_noun_error,0.996023,replace:delete->to lead,high_impact,False,True
2353,calendar_query,calendar,please tell me the pending reminders,PLEASE TELL ME TO PENDING REMIND US,PLEASE TEL ME THE PENDING REMINDERS,proper_noun_error,0.995963,replace:the->to | replace:reminders->remind us,high_impact,False,True
7914,calendar_remove,calendar,delete meeting,MEET MEETIN,DEMETE METING,proper_noun_error,0.995812,replace:delete meeting->meet meetin,high_impact,False,True
2252,qa_factoid,qa,the following question asks you to analyze tea...,THE FOLLOWING QUESTION ASKED YOU TO ANALYZE TE...,THE FOLOWING QUESTION ASK YOU TO ANALYZE TEACH...,proper_noun_error,0.995575,replace:asks->asked | replace:teacher goals->t...,high_impact,False,False
6845,lists_createoradd,lists,remind me to order more soap,REMIND ME TO OLL DE MORESOUP,REMIND ME TO ORDER MORE SOAP,proper_noun_error,0.995489,replace:order more soap->oll de moresoup,high_impact,False,False
3222,social_query,social,what was the last thing my mom said,WHAT WAS LOT I THINK MY MOUSICK,WHAT WOES THE LAST TAKE MY MOUTH SIAK,proper_noun_error,0.995327,replace:the last thing->lot i think | replace:...,high_impact,False,False
2254,qa_factoid,qa,the following question asks you to analyze tea...,THE FOLLOWING QUESTION ASKED YOU TO ANALYZE TE...,THE FOLOWING QUESTION ASKED YOU TO ANALYZE TEA...,proper_noun_error,0.995241,replace:asks->asked | replace:teacher goals->t...,high_impact,False,False


## Cell 20 — Save Notebook 15 outputs


In [20]:
risk_df.to_csv(REPORTS_DIR / "semantic_error_risk_dataset.csv", index=False)
model_comparison_df.to_csv(REPORTS_DIR / "semantic_error_risk_model_comparison.csv", index=False)
cv_summary_df.to_csv(REPORTS_DIR / "semantic_error_risk_cv_summary.csv", index=False)
bootstrap_significance_df.to_csv(REPORTS_DIR / "semantic_error_risk_bootstrap_significance.csv", index=False)
logreg_feature_importance_baseline_df.to_csv(REPORTS_DIR / "semantic_error_risk_feature_importance_logreg_baseline.csv", index=False)
logreg_feature_importance_finetuned_df.to_csv(REPORTS_DIR / "semantic_error_risk_feature_importance_logreg_finetuned.csv", index=False)
rf_feature_importance_baseline_df.to_csv(REPORTS_DIR / "semantic_error_risk_feature_importance_rf_baseline.csv", index=False)
rf_feature_importance_finetuned_df.to_csv(REPORTS_DIR / "semantic_error_risk_feature_importance_rf_finetuned.csv", index=False)
semantic_error_risk_score_bins_df.to_csv(REPORTS_DIR / "semantic_error_risk_score_bins.csv", index=False)
intent_risk_baseline_supported_df.to_csv(REPORTS_DIR / "intent_risk_baseline.csv", index=False)
intent_risk_finetuned_supported_df.to_csv(REPORTS_DIR / "intent_risk_finetuned.csv", index=False)
high_confidence_wrong_df.to_csv(REPORTS_DIR / "semantic_error_risk_case_studies.csv", index=False)

summary = {
    "best_holdout_model": model_comparison_df.iloc[0]["model_name"],
    "best_target": model_comparison_df.iloc[0]["target"],
    "best_holdout_roc_auc": float(model_comparison_df.iloc[0]["roc_auc"]),
    "best_holdout_pr_auc": float(model_comparison_df.iloc[0]["pr_auc"]),
    "methodological_note": "Taxonomy features in this notebook are post-hoc and ground-truth-aware; this is not yet a deployable real-time risk predictor.",
}
with open(REPORTS_DIR / "semantic_error_risk_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved Notebook 15 outputs.")


Saved Notebook 15 outputs.


## Cell 21 — Final conclusion and limitations


In [21]:
print("=" * 72)
print("SEMANTIC ERROR RISK PREDICTION SUMMARY")
print("=" * 72)
print("Model comparison:")
print(model_comparison_df[["target", "model_name", "roc_auc", "pr_auc", "f1", "brier"]].to_string(index=False))
print()
print("Cross-validation summary:")
print(cv_summary_df.to_string(index=False))
print()
print("Paired bootstrap CI for AUC difference (combined-full vs proxy-only):")
print(bootstrap_significance_df.to_string(index=False))
print()
print("Methodological limitation:")
print("- Taxonomy features here are post-hoc and require ground-truth-aware alignment.")
print("- This notebook therefore supports explanatory/predictive analysis, not deployment-ready risk scoring.")
print("- A future deployable model should use ASR-native uncertainty signals only.")
print()
print("Saved outputs to reports/ including:")
print("- semantic_error_risk_dataset.csv")
print("- semantic_error_risk_model_comparison.csv")
print("- semantic_error_risk_cv_summary.csv")
print("- semantic_error_risk_bootstrap_significance.csv")
print("- semantic_error_risk_feature_importance_*.csv")
print("- semantic_error_risk_score_bins.csv")
print("- intent_risk_baseline.csv")
print("- intent_risk_finetuned.csv")
print("- semantic_error_risk_case_studies.csv")
print("- semantic_error_risk_summary.json")


SEMANTIC ERROR RISK PREDICTION SUMMARY
Model comparison:
                 target                     model_name  roc_auc   pr_auc       f1    brier
 baseline_intent_failed           logreg_combined_full 0.840839 0.880966 0.824666 0.154544
 baseline_intent_failed logreg_combined_no_interaction 0.840827 0.880949 0.824666 0.154550
 baseline_intent_failed               rf_combined_full 0.840751 0.873598 0.823422 0.159107
 baseline_intent_failed     rf_combined_no_interaction 0.840619 0.874126 0.820183 0.159394
 baseline_intent_failed           logreg_taxonomy_only 0.805753 0.848710 0.808472 0.175475
 baseline_intent_failed              logreg_proxy_only 0.795079 0.846445 0.796595 0.179508
 baseline_intent_failed                  rf_proxy_only 0.784072 0.834469 0.778499 0.183649
 baseline_intent_failed               rf_taxonomy_only 0.755600 0.802597 0.754159 0.225815
 baseline_intent_failed         logreg_random_baseline 0.492190 0.603185 0.761667 0.237159
finetuned_intent_failed     rf_co

## Conclusion

This notebook established — under ground-truth-aware (post-hoc) analysis — that semantic taxonomy features carry predictive signal for downstream intent failure beyond ASR-style proxy metrics (edit counts, lexical overlap) alone.

**Key results**
- Combined-vs-proxy AUC gain (paired bootstrap, logistic regression): **+0.0459** (95% CI [0.0328, 0.0597]) on the baseline-ASR target and **+0.1340** (95% CI [0.1090, 0.1563]) on the fine-tuned-ASR target — both entirely positive (`fraction_positive` = 1.0).
- 5-fold CV ROC-AUC on the fine-tuned target: proxy 0.699 → taxonomy 0.817 → combined 0.834.
- Best holdout ROC-AUC 0.841 (baseline, `logreg_combined_full`) and 0.915 (fine-tuned, `rf_combined_no_interaction`); the random-feature baseline at 0.492 confirms the signal is not an artifact.

**Caveat:** the taxonomy features are **post-hoc and ground-truth-aware** — derived from alignment against the reference transcript — so they are **not available to a deployed system at inference time**. This notebook is explanatory, not a deployable risk predictor.

**Feeds into:** `16_reference_free_voxintel_risk_prediction.ipynb`, which redoes risk prediction with a strict leakage-free feature set (ASR-native + intent-native inference-time signals only).